# poblar en Colab

Calcula la población expuesta a la degradación de tierras a partir de un censo por radio, una cobertura del suelo y un mapa de degradación. Todos los criterios están en el archivo de configuración; los productos, los controles y el registro de cada ejecución quedan en la carpeta de salida.

1. Copiar los insumos a una carpeta de Drive (celda 3, `INSUMOS`).
2. Elegir la configuración y las etapas (celda 3).
3. Ejecutar las celdas en orden. Revisar los controles (celda 5).
4. Con dos años ejecutados, unir las tablas (celda 7).

Entorno: Colab (CPU). Los rasters se procesan en el disco local (`/content/trabajo`) y lo terminado se copia a Drive a medida que avanza; si la sesión se corta, volver a ejecutar con las etapas que faltan.


In [ ]:
# 1. Instalación
# Se clona el repositorio y se instala desde el clon: el código y las configuraciones
# son de la misma versión, y la ejecución registra el commit en entorno.txt.
# Para reproducir una versión exacta, indicar una etiqueta o un commit en VERSION.
VERSION = 'main'
!rm -rf /content/poblar_repo
!git clone -q https://github.com/AugustoBellanich/poblar /content/poblar_repo
!git -C /content/poblar_repo checkout -q {VERSION}
!pip install -q -e /content/poblar_repo
# La instalación editable se activa recién al reiniciar el intérprete; con esto
# la librería se puede importar en la misma sesión, sin reiniciar.
import sys, importlib
if '/content/poblar_repo' not in sys.path:
    sys.path.insert(0, '/content/poblar_repo')
importlib.invalidate_caches()
# Si en esta sesión ya se había importado una versión anterior, se descarta para usar la nueva.
for _m in [m for m in sys.modules if m == 'poblar' or m.startswith('poblar.')]:
    del sys.modules[_m]
!git -C /content/poblar_repo log -1 --format='Versión: %h — %s (%cs)'

# Herramientas de GDAL (gdalwarp, gdal_rasterize) para las etapas 2 y 3
!which gdalwarp > /dev/null || (apt-get -qq update && apt-get -qq install -y gdal-bin > /dev/null)
!gdalinfo --version


In [ ]:
# 2. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Parámetros de la ejecución
from pathlib import Path

ANIO    = 2010                                                   # 2010 o 2022
APLICACION = 'prais2026'                                       # carpeta en aplicaciones/
CONFIG  = Path(f'/content/poblar_repo/aplicaciones/{APLICACION}/configs/{ANIO}.yaml')
INSUMOS = Path('/content/drive/MyDrive/PRAIS/poblar/insumos')   # tabla_censoAAAA.xlsx, radios_AAAA.zip, regiones_PRAIS.gpkg, MB_prais_2015_100m.tif (2010) o MB_prais_2023_100m.tif (2022),
                                                                 # degradacion_referencia_2010.tif o degradacion_reporte_2022.tif
SALIDA  = Path('/content/drive/MyDrive/PRAIS/poblar/ejecuciones')  # una subcarpeta por ejecución
ETAPAS  = ['insumos', 'radios', 'grilla', 'dasimetria', 'cruce', 'publicacion', 'tablas']
# Para retomar una ejecución existente, indicar solo las etapas que faltan (por ejemplo ['tablas']).
TRABAJO = Path('/content/trabajo')                              # disco local de Colab; al terminar se copia a SALIDA

# Verificación previa: que estén los insumos que pide la configuración
import yaml
cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
faltan = [i['ruta'] for i in cfg['insumos'].values() if not (INSUMOS / i['ruta']).exists()]
if faltan:
    raise FileNotFoundError(f'Faltan insumos en {INSUMOS}: {faltan}')
print('Insumos completos:', [i['ruta'] for i in cfg['insumos'].values()])

In [ ]:
# 4. Ejecución
# Si un control bloqueante falla, la ejecución se detiene y se informa el motivo;
# los controles y el balance quedan igual en la carpeta para revisarlos.
from poblar import ejecutar
from poblar.trazabilidad import ControlFallido

try:
    ejecucion = ejecutar(CONFIG, insumos=INSUMOS, salida=SALIDA, etapas=ETAPAS,
                         trabajo=TRABAJO)
    CARPETA = ejecucion.carpeta
    print('\nEjecución completa:', CARPETA)
except ControlFallido as e:
    CARPETA = e.carpeta
    print('\nLa ejecución se detuvo:', e)
    print('Carpeta:', CARPETA)

In [ ]:
# 5. Controles y balance
import pandas as pd
pd.set_option('display.max_colwidth', 120)
controles = pd.read_csv(CARPETA / 'controles.csv')
print(f"Controles: {len(controles)} | sin cumplir: {(~controles['ok']).sum()}")
display(controles[['etapa', 'nombre', 'ok', 'detalle', 'bloqueante']])
display(pd.read_csv(CARPETA / 'balance.csv'))

## Resultados

Población expuesta (nacional y por región) y rasters publicados de la ejecución.


In [ ]:
# 6. Resultados de la ejecución
for archivo in ['4_cruce/resumen_expuesta.csv', '5_publicacion/publicacion.csv']:
    ruta = CARPETA / archivo
    if ruta.exists():
        print(archivo); display(pd.read_csv(ruta))


## Tablas de dos períodos

Con la etapa `tablas` ejecutada en los dos años, une las tablas con una columna por período.


In [ ]:
# 7. Tablas de dos períodos
from poblar.tablas import unir_periodos

ANIOS = (2010, 2022)
EJEC = {a: SALIDA / f'{APLICACION}_{a}' for a in ANIOS}
DEST = SALIDA / f'{APLICACION}_tablas'
for nombre, t in unir_periodos(EJEC, DEST).items():
    print(nombre); display(t)
print('Tablas en', DEST)
